# 경쟁사 비교 노드

`node.py`의 입출력 계약과 PDF의 경쟁 제품 비교표를 기준으로 정리한 코드입니다. 첫 셀이 프로젝트 루트의 고정 경로를 import 경로에 추가합니다. 셀을 위에서부터 실행하세요. 실제 그래프는 여전히 `node.py`의 `run`을 사용합니다.

## 1. 프로젝트의 타입과 공용 함수

In [ ]:
import json
import sys
from typing import Any

from pydantic import BaseModel, Field
from langchain_core.exceptions import OutputParserException

from core.llm import get_llm
from core.state import AnalysisResult, Candidate, GraphState, evaluation_update, make_analysis
from tools.web_search import WebResult, web_search

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool

tools = []

In [ ]:
"""테스트용 데이터"""
test_state: GraphState = {
    "query": "SK하이닉스 HBM 경쟁 제품 비교",
    "candidates": [
        {
            "company_id": "SK_HYNIX",
            "company_name": "SK하이닉스",
            "description": (
                "DRAM, HBM, NAND 플래시 등 메모리 반도체를 개발·생산하며 AI 서버용 고대역폭 메모리를 공급하는 기업"
            ),
            "domain": "HBM 메모리 반도체",
            "retrieval_score": 0.0,
        }
    ],
    "candidate_index": 1,
    "current_candidate": "SK_HYNIX",
    "evaluations": {},
    "selected_candidate": None,
    "report": None,
}

## 2. 경쟁 제품 검색

후보 정보로 검색어 하나를 만들어 `web_search`를 한 번 호출합니다. 반환값은 검색 결과인 `WebResult` 목록입니다.

In [ ]:
@tool
def search_competitors(company: Candidate, max_results: int = 5) -> list[WebResult]:
    """현재 기업과 같은 분야의 경쟁 제품 자료를 웹에서 검색한다."""
    name = company["company_name"].strip()
    domain = company["domain"].strip()
    query = f"{name} {domain} 경쟁 제품 비교 성능 전력 가격 고객"
    return web_search(query, max_results=max_results)

#search_competitors(test_state["candidates"][0])
tools = [search_competitors]

## 3. 분석 지침

In [29]:
COMPETITION_INSTRUCTIONS = """\
현재 기업과 같은 고객 문제를 해결하는 경쟁 제품, 경쟁사를 찾으세요.
평가 기업과 경쟁 제품의 기업·제품명, 용도·고객, 성능·전력, 가격·도입 조건, 강점·약점, 출처를 비교하세요.
성능을 비교할 때 모델, 연산 정밀도, 배치 크기, 측정 환경과 전력 측정 범위를 기록하세요.
비교 조건이 다르거나 독립 검증이 없으면 우열을 확정하지 마세요.
확인되지 않은 수치와 고객 관계를 추정하지 말고 missing_information에 기록하세요.
투자 점수는 산정하지 마세요.
summary, strengths, risks, evidence, details를 반환하세요.
details에는 peer_products, comparison_conditions, missing_information, critical_risks를 포함하세요.
peer_products의 각 항목에는 company_name, product_name, customer_and_use, performance_and_power,
price_and_adoption, strengths_and_weaknesses, source_url을 포함하세요.
evidence와 source_url에는 제공된 검색 결과의 URL만 설명 없이 그대로 사용하세요.
"""

## 4. 분석 결과의 구조와 검증

In [30]:
class PeerProduct(BaseModel):
    company_name: str = ""
    product_name: str = ""
    customer_and_use: str = ""
    performance_and_power: str = ""
    price_and_adoption: str = ""
    strengths_and_weaknesses: str = ""
    source_url: str = ""


class CompetitionDetails(BaseModel):
    peer_products: list[PeerProduct] = Field(default_factory=list)
    comparison_conditions: list[str] = Field(default_factory=list)
    missing_information: list[str] = Field(default_factory=list)
    critical_risks: list[str] = Field(default_factory=list)


class CompetitionResult(BaseModel):
    summary: str
    strengths: list[str]
    risks: list[str]
    evidence: list[str]
    details: CompetitionDetails


class SearchDecision(BaseModel):
    search_again: bool
    query: str
    reason: str

In [31]:
def _validate_result(
    result: AnalysisResult, sources: list[WebResult]
) -> None:
    """분석 결과의 형태와 출처 연결을 검사한다."""
    expected = {
        "summary": str,
        "strengths": list,
        "risks": list,
        "evidence": list,
        "details": dict,
    }
    for field, field_type in expected.items():
        if field not in result or not isinstance(result[field], field_type):
            raise ValueError(f"competition.{field}의 형식이 올바르지 않습니다.")

    if not all(isinstance(item, str) for item in result["evidence"]):
        raise ValueError("competition.evidence는 문자열 목록이어야 합니다.")

    details = result["details"]
    for field in (
        "peer_products",
        "comparison_conditions",
        "missing_information",
        "critical_risks",
    ):
        if field not in details:
            raise ValueError(f"competition.details.{field}가 없습니다.")

    source_urls = {
        source["url"] for source in sources if isinstance(source.get("url"), str) and source["url"]
    }
    for evidence in result["evidence"]:
        if evidence not in source_urls:
            raise ValueError(f"검색 출처로 확인되지 않은 근거입니다: {evidence}")

    for peer in details["peer_products"]:
        if peer["source_url"] not in source_urls:
            raise ValueError(f"검색 출처로 확인되지 않은 경쟁 제품입니다: {peer['source_url']}")

    if result["strengths"] or details["peer_products"]:
        if not result["evidence"]:
            raise ValueError("경쟁 우위나 경쟁 제품을 제시하려면 근거가 필요합니다.")

In [38]:
model = init_chat_model(
    model="gpt-6-luna",
    model_provider="openai",
    use_responses_api=True,
    reasoning_effort="low",
    max_tokens=1000, # 최대 생성 토큰수
    timeout=30,      # 요청 타임아웃(초)
)
agent = create_agent(
    model=model,
    tools=tools
)

PydanticUserError: Please use `typing_extensions.TypedDict` instead of `typing.TypedDict` on Python < 3.12.

For further information visit https://errors.pydantic.dev/2.13/u/typed-dict-version

## 5. 검색을 재시도하는 경쟁사 분석 에이전트

검색 자료가 없거나 분석 근거가 부족하면 모델이 새 검색어와 재검색 여부를 정합니다. 최대 3회 검색하고, 설정·스키마 오류는 그대로 보고합니다.

In [33]:
MAX_SEARCH_ATTEMPTS = 3


def _choose_next_search(company: Candidate, sources: list[WebResult], issue: str) -> SearchDecision:
    """현재 근거와 검증 오류를 보고 다음 검색어를 결정한다."""
    source_list = [{"title": item["title"], "url": item["url"]} for item in sources]
    prompt = (
        f"기업: {json.dumps(company, ensure_ascii=False)}\n"
        f"현재 문제: {issue}\n"
        f"이미 확보한 출처: {json.dumps(source_list, ensure_ascii=False)}"
    )
    return get_llm().with_structured_output(SearchDecision).invoke([
        ("system", "경쟁 제품의 근거를 보완할 새 웹 검색어를 정하세요. 추가 검색이 유용하면 search_again=true와 구체적인 query를, 검색으로 해결할 수 없으면 search_again=false를 반환하세요."),
        ("human", prompt),
    ])


def run(state: GraphState) -> dict:
    """현재 후보를 조사하고 근거가 부족하면 검색을 최대 3회 수행한다."""
    if CompetitionDetails.model_fields["comparison_conditions"].annotation != list[str]:
        raise RuntimeError("이전 스키마가 남아 있습니다. 4번 스키마 셀을 다시 실행하세요.")
    cid = state["current_candidate"]
    if not isinstance(cid, str) or not cid:
        raise ValueError("current_candidate에 기업 ID가 필요합니다.")

    company = next(
        (candidate for candidate in state["candidates"] if candidate["company_id"] == cid),
        None,
    )
    if company is None:
        raise ValueError(f"후보 목록에서 {cid}를 찾지 못했습니다.")

    sources: list[WebResult] = []
    used_queries: set[str] = set()
    next_query: str | None = None
    valid_result: AnalysisResult | None = None
    issue = "비교 가능한 자료가 부족합니다."

    for attempt in range(MAX_SEARCH_ATTEMPTS):
        if next_query is None:
            found = search_competitors(company)
        else:
            used_queries.add(next_query)
            found = web_search(next_query)
        sources.extend(found)

        if sources:
            prompt = (
                f"평가 기업: {json.dumps(company, ensure_ascii=False)}\n"
                f"검색 결과: {json.dumps(sources, ensure_ascii=False)}"
            )
            try:
                response = get_llm().with_structured_output(CompetitionResult).invoke(
                    [("system", COMPETITION_INSTRUCTIONS), ("human", prompt)]
                )
                result = response.model_dump()
                _validate_result(result, sources)
            except (ValueError, OutputParserException) as exc:
                issue = f"분석 결과 검증 실패: {exc}"
            else:
                valid_result = result
                missing = result["details"]["missing_information"]
                if not missing:
                    break
                issue = f"추가 확인이 필요한 정보: {json.dumps(missing, ensure_ascii=False)}"
        else:
            issue = "비교 가능한 검색 결과가 없습니다."

        if attempt + 1 >= MAX_SEARCH_ATTEMPTS:
            break
        decision = _choose_next_search(company, sources, issue)
        next_query = decision.query.strip()
        if not decision.search_again or not next_query or next_query in used_queries:
            break

    if valid_result is None:
        valid_result = make_analysis(
            summary="검증 가능한 경쟁 제품 비교 결과를 확보하지 못했습니다.",
            risks=[issue],
            details={
                "peer_products": [],
                "comparison_conditions": [],
                "missing_information": [issue],
                "critical_risks": [],
            },
        )
    return evaluation_update(cid, competition=make_analysis(**valid_result))

## 6. 테스트 실행

이 셀은 실제 웹 검색과 LLM을 호출합니다. 앞선 셀을 실행한 뒤 결과를 확인하세요.

In [34]:
run(test_state)

{'evaluations': {'SK_HYNIX': {'competition': {'summary': 'SK하이닉스의 HBM3E와 직접 경쟁하는 제품은 삼성전자 HBM3E 및 마이크론 HBM3E다. 제공된 비교 자료는 SK하이닉스와 삼성의 핀 속도·스택 대역폭 수치를 비슷한 수준으로 제시하고, 마이크론은 36GB 12단 제품의 대역폭과 전력 절감 주장을 제시한다. 다만 비교 조건과 독립적인 전력 측정이 확인되지 않아 성능 우열은 확정할 수 없다. 가격은 공개적으로 확인되지 않으며, 고객 인증·양산 물량과 실제 도입 조건도 제품별로 제한적으로만 확인된다.',
    'strengths': ['SK하이닉스는 제공된 기사에서 HBM3·HBM3E 공급 경험과 AI 반도체 업체에 대한 공급 확대가 언급되어 있다.',
     '비교 자료상 SK하이닉스 HBM3E의 핀 속도는 9.2Gb/s, 스택 대역폭은 1.15~1.2TB/s로 제시되어 경쟁 제품과 직접 비교 가능한 기준이 있다.',
     'HBM은 AI 가속기용 메모리로서 대역폭과 전력 효율, 고객 인증 및 안정적인 대량 공급이 중요한 구매 요소다.'],
    'risks': ['삼성전자와 마이크론이 HBM3E 및 차세대 HBM 공급을 확대하면 고객 인증과 공급 안정성을 둘러싼 경쟁이 심화될 수 있다.',
     '마이크론의 전력 절감 수치는 회사 주장 또는 출처의 요약이며, 동일 용량·적층 수·워크로드 조건에서 독립 검증된 비교 결과가 확인되지 않는다.',
     '제공된 검색 결과에는 실제 계약 가격, 고객별 도입 물량, 수율 및 제품별 전력 측정값이 충분히 공개되어 있지 않다.'],
    'evidence': ['https://www.datastorageic.com/detail/138.html',
     'https://www.techradar.com/pro/micron-launched-36gb-hbm3e-memory-as-it-plays-catch-up-with-samsung-and-sk-hyni

In [35]:
import core.config
print(repr(core.config.LLM_MODEL))
print(repr(get_llm().model_name))

'gpt-6-luna'
'gpt-6-luna'


In [36]:
print(CompetitionDetails.model_fields["comparison_conditions"].annotation)

list[str]


In [49]:
# 검색 도구를 직접 사용하는 에이전트 실행 셀
from langchain.tools import tool

def run_with_agent(state: GraphState) -> dict:
    cid = state["current_candidate"]
    if not isinstance(cid, str) or not cid:
        raise ValueError("current_candidate에 기업 ID가 필요합니다.")
    company = next(
        (item for item in state["candidates"] if item["company_id"] == cid),
        None,
    )
    if company is None:
        raise ValueError(f"후보 목록에서 {cid}를 찾지 못했습니다.")

    sources: list[WebResult] = []
    search_calls = 0
    accepted_result: dict | None = None

    @tool
    def competitor_web_search(query: str) -> str:
        """경쟁 제품과 경쟁사 자료를 웹에서 검색한다."""
        nonlocal search_calls
        if search_calls >= 3:
            return "검색 한도에 도달했습니다. 확보한 출처로 결과를 작성하세요."
        search_calls += 1
        try:
            found = web_search(query, max_results=5)
        except Exception as exc:
            return f"검색 연결 오류 ({type(exc).__name__}). 다른 검색어로 다시 시도하세요."
        sources.extend(found)
        return json.dumps(found, ensure_ascii=False)

    @tool
    def submit_competition_result(result: CompetitionResult) -> str:
        """경쟁 제품 비교 결과를 제출하고 검색 출처를 검증한다."""
        nonlocal accepted_result
        candidate_result = CompetitionResult.model_validate(result).model_dump()
        try:
            _validate_result(candidate_result, sources)
        except ValueError as exc:
            return f"검증 실패: {exc}. 검색 출처를 확인하고 결과를 수정해 다시 제출하세요. 이 과정은 summary에 쓰지 마세요."
        accepted_result = candidate_result
        return "검증 통과. 분석이 완료되었습니다."

    agent_model = init_chat_model(
        model="gpt-6-luna", model_provider="openai",
        use_responses_api=True, reasoning_effort="low",
        max_tokens=4000, timeout=30,
    )
    research_agent = create_agent(
        model=agent_model,
        tools=[competitor_web_search, submit_competition_result],
        system_prompt=(
            COMPETITION_INSTRUCTIONS
            + "\n결과 항목의 의미를 지키세요. "
            + "summary: 검색 근거에 기반해 분석 대상 기업과 경쟁사 제품을 비교한 핵심 결론을 요약하세요. "
            + "strengths: 경쟁사 제품과 비교했을 때 분석 대상 기업의 확인된 강점을 적으세요. "
            + "risks: 경쟁사 제품 때문에 분석 대상 기업이 직면하는 경쟁 위험과 불확실성을 적으세요. "
            + "evidence: 분석에 실제로 사용한 검색 결과의 URL만 문자열 목록으로 적으세요. "
            + "details.peer_products: 경쟁사별 제품, 용도, 성능·전력, 가격·도입, 장단점과 출처를 적으세요. "
            + "details.comparison_conditions: 공정한 비교에 필요한 제품 세대, 용량, 측정 조건을 적으세요. "
            + "details.missing_information: 근거가 없어 확인하지 못한 정보를 적으세요. "
            + "details.critical_risks: 분석 결론에 큰 영향을 주는 핵심 경쟁 위험을 적으세요. "
            + "\n먼저 검색 도구로 근거를 확보하세요. 검색은 최대 3회 가능합니다. "
            + "분석 대상 기업은 peer_products에서 제외하고 경쟁사 제품만 넣으세요. "
            + "evidence와 source_url에는 검색 도구가 반환한 URL만 넣으세요. "
            + "결과는 반드시 submit_competition_result 도구로 제출하세요. "
            + "검증에 실패하면 오류에 맞춰 재검색 또는 수정 후 다시 제출하세요. "
            + "summary에는 기업 비교 결론만 쓰고 검증 과정이나 도구 오류를 언급하지 마세요."
        ),
    )
    research_agent.invoke(
        {"messages": [{"role": "user", "content": (
            f"다음 기업의 경쟁 제품을 비교하세요: {json.dumps(company, ensure_ascii=False)}"
        )}]},
        config={"recursion_limit": 24},
    )

    if accepted_result is None:
        raise RuntimeError("에이전트가 검증을 통과한 분석 결과를 제출하지 않았습니다.")
    return evaluation_update(cid, competition=make_analysis(**accepted_result))

print(json.dumps(run_with_agent(test_state), ensure_ascii=False, indent=2))

{
  "evaluations": {
    "SK_HYNIX": {
      "competition": {
        "summary": "SK하이닉스의 12단 HBM3E는 36GB 제품의 양산 개시가 공식 발표되어 비교 대상 중 상용화 시점이 확인되는 강점이 있다. 삼성전자도 36GB·12단과 최대 1,280GB/s를 발표했고, 마이크론은 24GB·8단 제품의 NVIDIA H200 채택 계획과 36GB·12단 샘플을 공개했다. 공개 자료만으로 동일 조건의 전력·성능 우열이나 가격 경쟁력을 확정할 수 없다.",
        "strengths": [
          "SK하이닉스는 36GB 12단 HBM3E의 양산을 시작했다고 밝혀, 경쟁사 자료의 개발 발표·샘플 가용성과 구별되는 양산 단계 근거가 있다.",
          "SK하이닉스는 12단 제품이 이전 8단 제품과 동일한 두께 규격을 유지한다고 발표했다."
        ],
        "risks": [
          "삼성전자와 마이크론도 36GB 12단 HBM3E 제품을 발표해 용량 측면의 차별성이 제한될 수 있다.",
          "마이크론은 HBM3E 24GB의 NVIDIA H200 탑재 출하 계획을 밝혀 고객 플랫폼 채택 경쟁이 확인된다.",
          "성능 및 전력 수치는 업체별 조건과 표기가 달라 직접 우열을 판단할 수 없고, 가격·수율·공급량 자료도 확인되지 않았다."
        ],
        "evidence": [
          "https://news.skhynix.com/en/sk-hynix-begins-volume-production-of-the-world-first-12-layer-hbm3e",
          "https://news.samsung.com/global/samsung-develops-industry-first-36gb-hbm3e-12h-dram",
          "https://www.micron.com/pr